In [35]:
from ast import *
from utils import *
from x86_ast import *
import import_ipynb
from rco_test import *
from select_instr import *
from assign_homes import *


In [36]:
def patch_instr(i: instr) -> List[instr]:
    match i:
        case Instr('movq',[Deref('rbp',num1),Deref('rbp',num2)]):
            return [Instr('movq',[Deref('rbp',num1),Reg('rax')]), 
                    Instr('movq',[Reg('rax'),Deref('rbp',num2)])]
        case Instr('addq',[Immediate(int), Deref('rbp',num)]):
            if Immediate(int).value > 2**16:
                return [Instr('movq',[Immediate(int), Reg('rax')]),
                        Instr('addq',[Reg('rax'),Deref('rbp',num)])]
            return [i]
        case Instr('subq',[Immediate(int),Deref('rbp',num)]):
            if Immediate(int).value > 2**16:
                return [Instr('movq',[Immediate(int),Reg('rax')]),
                        Instr('subq',[Reg('rax'),Deref('rbp',num)])]
            return [i]
        case _:
            return [i]

In [37]:
def patch_instructions(p: X86Program) -> X86Program:
    new_list = []
    for instr in p.body:
                  new_list.extend(patch_instr(instr))
    return X86Program(new_list)

In [38]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    a = 12
    b = a 
    print(b)""")
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_code = select_instruction(rco_code)
    assign_homes_code = assign_homes(select_instr_code)
    patch_instr_code = patch_instructions(assign_homes_code)
    print(select_instr_code)
    print(assign_homes_code)
    print(patch_instr_code)

	.globl main
main:
    movq $12, a
    movq a, b
    movq b, %rdi
    callq print_int


	.globl main
main:
    movq $12, -8(%rbp)
    movq -8(%rbp), -16(%rbp)
    movq -16(%rbp), %rdi
    callq print_int


	.globl main
main:
    movq $12, -8(%rbp)
    movq -8(%rbp), %rax
    movq %rax, -16(%rbp)
    movq -16(%rbp), %rdi
    callq print_int


